In [14]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

import joblib
import os

In [15]:
# Load the cleaned dataset created in Notebook 2.

input_path = "../data/processed/yelp_balanced_150k_cleaned.csv"

df = pd.read_csv(input_path)

print("Dataset shape:", df.shape)

Dataset shape: (150000, 6)


In [16]:
df.head()

,class_index,review_text,sentiment,review_length,word_count,cleaned_text
0,3,My friends and I were looking for a decent but...,Neutral,609,108,friend looking decent relatively inexpensive c...
1,1,Not the greatest steaks...have had much better...,Negative,1132,205,not_the greatest steakshave much better servic...
2,5,Wow. I'm so impressed. Service was great. Info...,Positive,404,75,wow im impressed service great informative pat...
3,5,Good stuff! I remember coming here years ago a...,Positive,787,141,good stuff remember coming year ago trying ron...
4,5,My friend and I decided to look at side tables...,Positive,539,94,friend decided look side table june 22 2014 ho...


In [17]:
print("Missing values:")
print(df.isnull().sum())

Missing values:
class_index      0
review_text      0
sentiment        0
review_length    0
word_count       0
cleaned_text     0
dtype: int64


In [19]:
# Check for empty strings in cleaned_text.

empty_cleaned_text = (
    df["cleaned_text"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print("Empty cleaned_text values:", empty_cleaned_text)

Empty cleaned_text values: 0


In [20]:
# Check the distribution of the target variable.

print("Sentiment distribution:")
print(df["sentiment"].value_counts())

Sentiment distribution:
sentiment
Neutral     50000
Negative    50000
Positive    50000
Name: count, dtype: int64


In [21]:
# X = input feature
# y = target variable

X = df["cleaned_text"]

y = df["sentiment"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (150000,)
y shape: (150000,)


In [22]:
# Split the dataset into training and testing sets.
# The split is performed BEFORE TF-IDF vectorization
# to prevent data leakage.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 120000
Testing samples: 30000


In [23]:
# Check sentiment distribution in the training data.

print("Training set distribution:")
print(y_train.value_counts())

print("\nTesting set distribution:")
print(y_test.value_counts())

Training set distribution:
sentiment
Positive    40000
Negative    40000
Neutral     40000
Name: count, dtype: int64

Testing set distribution:
sentiment
Neutral     10000
Negative    10000
Positive    10000
Name: count, dtype: int64


In [24]:
# Create the TF-IDF vectorizer.

tfidf_vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

print("TF-IDF vectorizer created.")

TF-IDF vectorizer created.


In [25]:
# Fit the TF-IDF vectorizer ONLY on the training data.
# Then transform the training data.

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

print("TF-IDF fitted on training data.")
print("Training TF-IDF shape:", X_train_tfidf.shape)

TF-IDF fitted on training data.
Training TF-IDF shape: (120000, 50000)


In [26]:
# Transform the test data using the TF-IDF vocabulary
# learned from the training data.

X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("Test data transformed successfully.")
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Test data transformed successfully.
Testing TF-IDF shape: (30000, 50000)


In [27]:
# Check the number of features learned by TF-IDF.

feature_names = tfidf_vectorizer.get_feature_names_out()

print("Number of TF-IDF features:", len(feature_names))

Number of TF-IDF features: 50000


In [28]:
# Display the first 30 TF-IDF features.

print(feature_names[:30])

['05' '050' '08' '09' '10' '10 10' '10 12' '10 15' '10 20' '10 buck'
 '10 cent' '10 coupon' '10 day' '10 different' '10 discount' '10 dollar'
 '10 drink' '10 foot' '10 get' '10 hour' '10 item' '10 mile' '10 min'
 '10 minute' '10 month' '10 oz' '10 people' '10 per' '10 piece' '10 place']


In [29]:
# Inspect the TF-IDF representation of the first training review.

first_review_vector = X_train_tfidf[0]

print("Number of non-zero TF-IDF values:")
print(first_review_vector.nnz)

Number of non-zero TF-IDF values:
31


In [30]:
# Display the original cleaned review.

print("First cleaned review:")
print(X_train.iloc[0])

First cleaned review:
two day christmas total chaos everywhere except service excellent food outstanding husband started mediterranean small bite pizza love whole wheat crust service fantastic


In [34]:
# Final verification of the Notebook 3 pipeline.

print("FINAL VERIFICATION ")

print("\nOriginal dataset:")
print("Rows:", len(df))

print("\nTraining data:")
print("Samples:", X_train_tfidf.shape[0])
print("Features:", X_train_tfidf.shape[1])

print("\nTesting data:")
print("Samples:", X_test_tfidf.shape[0])
print("Features:", X_test_tfidf.shape[1])

print("\nTraining labels:")
print(y_train.value_counts())

print("\nTesting labels:")
print(y_test.value_counts())

FINAL VERIFICATION 

Original dataset:
Rows: 150000

Training data:
Samples: 120000
Features: 50000

Testing data:
Samples: 30000
Features: 50000

Training labels:
sentiment
Positive    40000
Negative    40000
Neutral     40000
Name: count, dtype: int64

Testing labels:
sentiment
Neutral     10000
Negative    10000
Positive    10000
Name: count, dtype: int64
